# Parking-Induced Congestion Intelligence — Analysis & Methodology

**Problem (HackerEarth PS-1):** detect illegal-parking hotspots and quantify their impact on traffic flow to enable targeted enforcement, using **only** the provided Bengaluru Traffic Police violations dataset.

This notebook is the *methodology artifact*: it runs EDA, justifies the Congestion-Impact Score weights, validates the spatial clustering, and previews the top enforcement hotspots. The validated logic lives in `backend/app/scoring.py` and `backend/app/clustering.py`, which the FastAPI dashboard reuses verbatim.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd().parent
sys.path.insert(0, str(PROJECT / 'backend'))
from app import scoring
from app.clustering import assign_clusters

RAW = PROJECT / 'data' / 'raw' / 'violations.csv'
pd.set_option('display.max_columns', 50)
print('reading', RAW)

## 1. Load & shape

In [ ]:
df = pd.read_csv(RAW, low_memory=False)
print(df.shape)
df.head(3)

In [ ]:
# Null audit — note closed_datetime / action_taken_timestamp are 100% empty (unusable)
null_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
null_pct

## 2. Feature engineering (mirrors the pipeline)

In [ ]:
def parse_types(raw):
    try:
        v = json.loads(raw)
        return [str(x).strip().upper() for x in v] if isinstance(v, list) else []
    except Exception:
        return []

df['violation_types'] = df['violation_type'].apply(parse_types)
df['severity_weight'] = df['violation_types'].apply(scoring.severity_weight_for)
df['vehicle_type'] = df['vehicle_type'].fillna('UNKNOWN').str.upper().str.strip()
df['vehicle_weight'] = df['vehicle_type'].apply(scoring.vehicle_weight_for)

created = pd.to_datetime(df['created_datetime'], errors='coerce', utc=True) + pd.Timedelta(hours=5, minutes=30)
df['hour'] = created.dt.hour
df['date'] = created.dt.date.astype(str)
df['is_peak'] = df['hour'].isin(scoring.PEAK_HOURS).astype(int)
df['at_junction'] = (df['junction_name'].fillna('No Junction').str.lower() != 'no junction').astype(int)
df['validation_status'] = df['validation_status'].fillna('unvalidated')
print('date range:', created.min(), '->', created.max())

## 3. EDA — what drives congestion

In [ ]:
vt = df['violation_types'].explode().value_counts()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
vt.head(10).iloc[::-1].plot.barh(ax=ax[0], color='#3b82f6'); ax[0].set_title('Violation types')
df['vehicle_type'].value_counts().head(10).iloc[::-1].plot.barh(ax=ax[1], color='#8b5cf6'); ax[1].set_title('Vehicle types')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df['hour'].value_counts().sort_index().plot.bar(ax=ax[0], color='#f59e0b'); ax[0].set_title('Violations by hour (IST)')
df['police_station'].value_counts().head(12).iloc[::-1].plot.barh(ax=ax[1], color='#10b981'); ax[1].set_title('Top stations')
plt.tight_layout()

## 4. Congestion-Impact Score — methodology

No traffic-speed feed exists in the data, so impact is a **proxy** combining five CSV-derived signals (weights in `scoring.SCORE_WEIGHTS`). The severity and vehicle weight tables encode the domain assumption that *blocking a main road with a bus hurts flow far more than a scooter in a no-parking bay*.

In [ ]:
print('Score component weights:', scoring.SCORE_WEIGHTS)
pd.Series(scoring.SEVERITY_WEIGHTS).sort_values(ascending=False).head(12)

## 5. Spatial clustering (DBSCAN, haversine ~50 m)

In [ ]:
df = df[df['latitude'].between(11.5, 14.5) & df['longitude'].between(76.5, 78.5)].copy()
df['cluster_id'] = assign_clusters(df, eps_m=50, min_samples=5)
n = df.loc[df.cluster_id >= 0, 'cluster_id'].nunique()
print(f'{n:,} clusters; {(df.cluster_id == -1).mean():.1%} noise')

In [ ]:
# Aggregate per cluster and score (same shape the API builds)
g = df[df.cluster_id >= 0].groupby('cluster_id')
clusters = pd.DataFrame({
    'count': g.size(),
    'active_days': g['date'].nunique(),
    'lat': g['latitude'].mean(), 'lon': g['longitude'].mean(),
    'mean_severity': g['severity_weight'].mean(),
    'mean_vehicle': g['vehicle_weight'].mean(),
    'junction_share': g['at_junction'].mean(),
    'peak_share': g['is_peak'].mean(),
    'top_station': g['police_station'].agg(lambda s: s.mode().iat[0] if len(s.mode()) else None),
}).reset_index()
clusters = scoring.compute_impact_scores(clusters).sort_values('impact_score', ascending=False)
clusters.head(15)[['cluster_id','impact_score','count','active_days','junction_share','top_station']]

**Sanity check:** the top-ranked hotspots should concentrate in the known heavy commercial stations (Upparpet, Shivajinagar, City Market, KR Market), and high-volume junction/main-road clusters should outrank equal-volume residential no-parking clusters.

## 6. Hotspot map (folium)

In [ ]:
import folium
top = clusters.head(150)
m = folium.Map(location=[12.97, 77.59], zoom_start=12, tiles='cartodbpositron')
def color(s):
    return '#ef4444' if s > 60 else '#f59e0b' if s > 40 else '#2ecc71'
for _, r in top.iterrows():
    folium.CircleMarker(
        [r.lat, r.lon], radius=4 + (r['count'] ** 0.5),
        color=color(r.impact_score), fill=True, fill_opacity=0.7,
        popup=f"score {r.impact_score} | {int(r['count'])} viol | {r.top_station}",
    ).add_to(m)
m